# Project Structure & CLI

*Notebook 30*

Notebooks are for learning and prototyping. When you're ready to turn an agent into something real — a script, a CLI tool, an API — a few structural changes make the difference between a demo and a deployable system.
<br>
<br>

**Topics:**

- What changes when you leave Jupyter

- A reusable project structure for agent apps

- Separating config, tools, and agent logic into modules

- Running an agent from a script or CLI

- Streaming agent output in a script

- When to drop down to `client.responses.create()` directly

- When to consider a simple API wrapper

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

print("✅ Ready!")

---

## 🎯 The Problem

Notebooks hide a lot of setup for you.

In a real script, you need to handle structure, async execution, and secrets explicitly.

---

## 📋 Part 1: What Changes When You Leave Jupyter

Three things that work effortlessly in notebooks become explicit concerns in real code:

**Top-level `await`** — JupyterLab handles this for you. In a script, you need `asyncio.run()` to run coroutines.

**Global state** — In notebooks, agents and tools defined in one cell are available everywhere. In a module-based project, you need to import them explicitly.

**Configuration** — `.env` loading works in notebooks but you need to decide where it happens in a real app — and ensure it runs before anything uses the API key.

---

## 🏗️ Part 2: A Reusable Project Structure

This structure works for most agent projects — from a simple CLI tool to a small web service:

```
my_agent_project/
├── .env                    # API key — never commit this
├── .gitignore              # include .env
├── requirements.txt        # openai-agents, python-dotenv, ...
│
├── config.py                # constants: MODEL, REASONING_MODEL, paths
├── tools.py                # all @function_tool definitions
├── agent.py                # Agent and Runner — creates and runs the agent
└── main.py                 # entry point — CLI, loop, or API handler
```

For larger projects, split `tools.py` into a `tools/` folder with one file per domain.

For a web service, `main.py` becomes a FastAPI or Flask app.

---

## 📄 Part 3: The Four Files

Here's what each file looks like.

These are printed templates you can copy directly into real project files — they are not executed in this notebook.

This matters because once your agent lives in files instead of notebook cells, you can run it from the terminal, import it into other apps, and change one part without touching the rest.

In [ ]:
# config.py — constants and environment loading
# -----------------------------------------------
CONFIG_TEMPLATE = '''
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path(__file__).parent / ".env")

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

# os.getenv() reads from environment — works for .env locally and platform secrets in production
# Never hardcode keys in this file. Add .env to .gitignore immediately.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not set — check your .env or platform secrets")

# Add project-specific constants here
# MAX_RETRIES = 3
# DB_PATH = Path(__file__).parent / "data" / "agent.sqlite"
'''

print("config.py:")
print(CONFIG_TEMPLATE)

#### tools.py

In [ ]:
# tools.py — all @function_tool definitions in one place
# -------------------------------------------------------
TOOLS_TEMPLATE = '''
from agents import function_tool


@function_tool
def my_tool(query: str) -> str:
    """One-line description of what this tool does."""
    try:
        return f"Result for: {query}"
    except Exception as e:
        return f"Error: {e}"


# Add more tools here — all imported into agent.py
'''

print("tools.py:")
print(TOOLS_TEMPLATE)

#### agent.py

In [ ]:
# agent.py — agent definition and async run function
# ---------------------------------------------------
AGENT_TEMPLATE = '''
from agents import Agent, Runner
from config import MODEL
from tools import my_tool


instructions = "Your agent instructions here."

agent = Agent(
    name="MyAgent",
    instructions=instructions,
    model=MODEL,
    tools=[my_tool]
)


async def run_agent(message: str) -> str:
    """Run the agent and return its response."""

    result = await Runner.run(agent, input=message)

    return result.final_output
'''

print("agent.py:")
print(AGENT_TEMPLATE)

#### main.py

In [ ]:
# main.py — CLI entry point
# --------------------------
MAIN_TEMPLATE = '''
import asyncio
from agent import run_agent


async def main():
    print("Agent ready. Type 'quit' to exit.")
    while True:
        message = input("\nYou: ").strip()
        if message.lower() in ["quit", "exit", "q"]:
            break
        if not message:
            continue
        response = await run_agent(message)
        print(f"Agent: {response}")


if __name__ == "__main__":
    asyncio.run(main())
'''

print("main.py:")
print(MAIN_TEMPLATE)

### 💡 Why This Structure Works

Each file has one responsibility, so adding a tool or chasing a bug always has an obvious home.

This is the simplest CLI-style entry point — for richer argument handling (flags, subcommands), layer `argparse` or a library like `typer` on top of the same structure.

---

## 📡 Part 4: Streaming Agent Output in a Script

In Jupyter, output appears when the cell finishes.

In a real script, you can stream tokens as they arrive — a much better experience for users waiting on a response.

Use `Runner.run_streamed()` instead of `Runner.run()`.

Same agent, same SDK, output appears live in the terminal.

In [ ]:
# main.py — streaming version
# ----------------------------
STREAMING_TEMPLATE = '''
import asyncio
from agents import Agent, Runner
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent
from agent import agent  # agent is already configured in agent.py — no extra imports needed


async def run_streaming(message: str) -> None:
    """Stream agent output token by token."""
    print("Agent: ", end="", flush=True)

    result = Runner.run_streamed(agent, input=message)

    async for event in result.stream_events():
        if isinstance(event, RawResponsesStreamEvent):
            data = event.data
            if isinstance(data, ResponseTextDeltaEvent):
                print(data.delta, end="", flush=True)

    print()  # newline after streaming completes


async def main():
    print("Agent ready (streaming). Type 'quit' to exit.")
    while True:
        message = input("\nYou: ").strip()
        if message.lower() in ["quit", "exit", "q"]:
            break
        if not message:
            continue
        await run_streaming(message)


if __name__ == "__main__":
    asyncio.run(main())
'''

print("main.py (streaming version):")
print(STREAMING_TEMPLATE)

### 💡 Why This Works

The agent definition is unchanged — same SDK, same tools.

What changes is how you consume the result: `Runner.run_streamed()` returns a stream you iterate over with `async for`, filtering for text-delta events.

---

## 🔽 Part 5: When to Drop Down to `client.responses.create()`

The Agents SDK is a higher-level library built on the core `openai` package — `client.responses.create()` is the lower-level API call underneath.

The Agents SDK handles orchestration, tools, handoffs, guardrails, and sessions.

Use it by default.

The decision is simple: if the model needs to decide what to do next, use an agent — if you already know the exact one-shot call you want, use the Responses API directly.

Drop down to `client.responses.create()` directly when you need a simple, single-turn model call with no agent machinery — tight latency requirements, fine-grained control over the request, or cases where the SDK's overhead genuinely isn't worth it.

In [ ]:
# Direct Responses API — use when you just need a model call
# ----------------------------------------------------------
RESPONSES_TEMPLATE = '''
from openai import OpenAI
from config import MODEL

client = OpenAI()  # reads OPENAI_API_KEY from the environment — config.py already loaded it


def classify(text: str) -> str:
    """Classify text with a direct model call — no agent overhead."""
    response = client.responses.create(
        model=MODEL,
        input=f"Classify this as positive, negative, or neutral: {text}"
    )
    return response.output_text
'''

print("Direct Responses API pattern:")
print(RESPONSES_TEMPLATE)

### 💡 When to Use Each

Use the **Agents SDK** when you need tools, handoffs, guardrails, sessions, or multi-step reasoning — anything where the agent needs to make decisions about what to do next.

Use **`client.responses.create()`** when you need a single, direct model call: classification, extraction, transformation, or any task where you're providing all the context and just want a response.

---

## 🌐 Part 6: When to Consider a Simple API Wrapper

This is an entry-point variation, not a deployment lesson — Lesson 32 covers deploying to a live environment.

A CLI loop works for personal tools.

Reach for an API wrapper when another process — a web frontend, a Slack bot, a scheduled job — needs to call your agent.

Add a thin API layer:

In [ ]:
# main.py — FastAPI version (requires: pip install fastapi uvicorn)
FASTAPI_TEMPLATE = '''
from fastapi import FastAPI
from pydantic import BaseModel
from agent import run_agent

app = FastAPI()


class ChatRequest(BaseModel):
    message: str


class ChatResponse(BaseModel):
    response: str


@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    response = await run_agent(request.message)
    return ChatResponse(response=response)


# Run with: uvicorn main:app --reload
# Call with: POST http://localhost:8000/chat
#              Body: {"message": "your question here"}
'''

print("main.py (FastAPI version):")
print(FASTAPI_TEMPLATE)

### 💡 Why This Works

`run_agent()` in `agent.py` is already `async` — FastAPI handles the event loop, so you just `await` it directly in the route handler.

The agent code doesn't change at all.

---

## 💪 Practice Exercises

### Exercise 1: Build the Four-File Structure

*Covers: project structure — the four-file agent template*

Create a working agent project using the templates from Part 3.

**Note:** For these exercises, work outside Jupyter — open a terminal, create the `my_agent/` directory, and create the `.py` files with a text editor, copying the templates from this notebook.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Build the Four-File Structure
# --------------------------------------------------------------
# Objective: Turn the templates from Part 3 into a working project.

# TODO 1: Create a project directory called my_agent/

# TODO 2: Create config.py using the CONFIG_TEMPLATE from Part 3

# TODO 3: Create tools.py with one @function_tool of your choice

# TODO 4: Create agent.py with an Agent that uses your tool
#           Use run_agent() as the async entry point

# TODO 5: Create main.py with the CLI loop from Part 3
#           Run it from the terminal: python main.py

# --- Write your code below this line ---

### Exercise 2: Add Streaming

*Covers: streaming output — modifying a project for streamed responses*

Modify your main.py from Exercise 1 to stream output using the pattern from Part 4.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Add Streaming
# --------------------------------------------------------------
# Objective: Replace Runner.run() with Runner.run_streamed() in main.py.

# TODO 1: Copy the streaming template from Part 4 into your main.py

# TODO 2: Run it from the terminal and observe tokens streaming live

# TODO 3: (Optional) Compare the experience to the non-streaming version

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Leaving Jupyter requires three explicit decisions:**

- Replace top-level `await` with `asyncio.run()` in `main.py`

- Import agents and tools explicitly — no more global notebook state

- Load `.env` in `config.py` before anything uses the API key
<br>
<br>

**Four-file structure separates concerns cleanly:**

- `config.py` owns constants, `tools.py` owns tools, `agent.py` owns the agent, `main.py` owns the entry point

- Same structure scales from a CLI tool to a web service
<br>
<br>

**Streaming keeps users engaged:**

- `Runner.run_streamed()` — same agent, output arrives token by token

- Filter `RawResponsesStreamEvent` → `ResponseTextDeltaEvent` to print text progressively
<br>
<br>

**Use the Agents SDK by default, drop to `client.responses.create()` when you just need a direct model call:**

- SDK for tools, handoffs, guardrails, sessions, multi-step reasoning

- Raw API for simple single-turn classification, extraction, or transformation
<br>
<br>

**Secrets travel through environment variables:**

- `.env` for local dev, platform secrets for deployment — same `os.getenv()` call either way

- Never hardcode, never commit `.env`
<br>
<br>

**A thin API wrapper is just another entry point:**

- Swap `main.py` from a CLI loop to a FastAPI app to expose your agent to other systems

- The agent code in `agent.py` doesn't change — only the entry point does

---

## 📍 Next Step

**Notebook 31: Architecture Decisions**  

A decision framework for model selection, single vs multi-agent, memory, MCP, and when to add complexity.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-30-project-structure-and-cli)

---